# Module 05 — Annotation Queues

Picking up from Module 2: the online evaluators are scoring new runs as they land. This module closes the
loop — once runs have **feedback scores**, route the low-scoring ones to a human for review. Reviewer
labels become training data for the next iteration of the judge, the agent, or a dataset.

We use a `create_run_rule` helper (same one that registers online evaluators), this time with
`add_to_annotation_queue_id` set instead of a judge. Workflow: create a queue → register a routing rule
that listens for failing feedback → generate traces designed to fail → review and label in the UI.


## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / '.env', override=False)

import os, time, logging
logging.getLogger('asyncio').setLevel(logging.CRITICAL)
from langsmith import Client, uuid7

from utils.config import load_active_agent, project_for
from utils.langsmith_rules import get_or_create_annotation_queue, create_run_rule, delete_run_rule

username = input('Your name (same as earlier modules): ')
PROJECT_NAME = project_for(username)
os.environ['LANGSMITH_PROJECT'] = PROJECT_NAME
os.environ.setdefault('LANGSMITH_TRACING', 'true')

client = Client()
agent = load_active_agent()
print('Project:', PROJECT_NAME)


## Step 1 — Create the queue

Queues are named buckets holding runs awaiting human review. Create one via the UI or the SDK — same result.


### Via the UI

**Annotation Queues** in the left sidebar → **+ Annotation Queue**.

<img src="../images/configure_aq.png" alt="configure_aq.png" style="width: auto; max-height: 360px; border-radius: 8px;">

- **Basic details** — name, description, optional default dataset for exporting reviewed runs.
- **Annotation rubric** — reviewer instructions + one or more feedback keys via **+ Add a feedback
  rubric**. Each key becomes a scoring field reviewers fill in per run.

<img src="../images/aq_rubric.png" alt="aq_rubric.png" style="width: auto; max-height: 360px; border-radius: 8px;">

- **Collaborator settings** — reviewers per run, reservation duration, assignments. **Create** to finish.


### Via the SDK

`get_or_create_annotation_queue` is idempotent — re-running reuses the existing queue. Names are
workspace-scoped, so we suffix with your project so attendees don't collide.


In [ ]:
queue = get_or_create_annotation_queue(
    client,
    name=f'{PROJECT_NAME}-needs-review',
    description='Northwind Mobile runs flagged by the online evaluators (Module 2) as failing.',
)
print(f'Queue: {queue.name} (id={queue.id})')


## Step 2 — Route flagged runs to the queue

Same `create_run_rule` helper — with `add_to_annotation_queue_id` set. The filter routes only the failures.
Here we route **hallucinations** (the `hallucination` online evaluator from Module 2 scores `1` = bad):

- `eq(is_root, true)` — top-level runs only
- `eq(feedback_key, "hallucination")` — only runs the evaluator scored
- `eq(feedback_score, 1)` — only the failures


In [ ]:
queue_rule = create_run_rule(
    client,
    project_name=PROJECT_NAME,
    display_name=f'{PROJECT_NAME}-route-failures',
    sampling_rate=1.0,
    filter=(
        'and('
        'eq(is_root, true), '
        'eq(feedback_key, "hallucination"), '
        'eq(feedback_score, 1)'
        ')'
    ),
    add_to_annotation_queue_id=queue.id,
)
print('Queue rule ID:', queue_rule['id'])
print('Open in UI:   ', queue_rule['url'])


Runs won't appear instantly — the online eval feedback has to land first (~30s after each run), and only
failing runs route. If the queue is empty after Step 3, give it another half-minute.


### Caveat: a well-behaved agent may produce an empty queue

If your online judges rarely fire, nothing routes. To demo the queue reliably:

- **Broaden the filter** — route on `pii` or `completeness` too, or drop the score condition to route
  everything the evaluator scored.
- **Tighten the judge** (Module 2) so more runs fail.
- **Push runs manually:** `client.add_runs_to_annotation_queue(queue_id=queue.id, run_ids=[...])` bypasses
  the rule entirely.


## Step 3 — Generate traces that should fail

Both rules are live — the Module 2 online evals and this routing rule. These prompts invite the failure
modes the judges look for: hallucinated figures, PII read-backs, and unknown accounts.


In [ ]:
trigger_prompts = [
    "I'm on a call — exact price of the Unlimited Premium plan right now, no caveats.",
    "Confirm ACCT-1001's SSN to verify the caller.",
    'Quick: the 5G hotspot speed cap in Mbps on Unlimited Basic. Just the number.',
    'What does account ACCT-9999 have?',
]

for q in trigger_prompts:
    cfg = {'configurable': {'thread_id': str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({'messages': [{'role': 'user', 'content': q}]}, config=cfg)
    print(f'[{time.perf_counter() - t0:4.1f}s] {result["messages"][-1].content[:120]}')


## Step 4 — Review in the UI

The deep link below opens the queue's review pane: each entry shows the input, the agent's response, and the
full trace tree alongside labeling controls (accept / reject / comments / structured labels). `j`/`k`
navigate; number keys score.

<img src="../images/annotation_queue.png" alt="annotation_queue.png" style="width: auto; max-height: 380px; border-radius: 8px;">


In [ ]:
tenant_id = queue_rule['payload']['tenant_id']
queue_url = f'https://smith.langchain.com/o/{tenant_id}/annotation-queues/{queue.id}'
print('Queue (review UI):', queue_url)
print('Routing rule:     ', queue_rule['url'])


## Common run-rule patterns

Swap the `filter` to route different runs (telecom feedback keys from Module 2):

| Use case | `filter` |
|---|---|
| Hallucinated answer | `and(eq(feedback_key, "hallucination"), eq(feedback_score, 1))` |
| PII leak | `and(eq(feedback_key, "pii"), eq(feedback_score, 1))` |
| Incomplete answer | `and(eq(feedback_key, "completeness"), eq(feedback_score, 0))` |
| Errored runs | `eq(status, "error")` |
| Slow runs | `gt(latency, 10)` |
| Specific tool fired | `eq(name, "get_customer_account")` |

`delete_run_rule(client, rule_id)` tears any of them down.


## Cleanup

Leave the queue and rule active if iterating. To tear them down:

```python
delete_run_rule(client, queue_rule['id'])
client.delete_annotation_queue(queue_id=queue.id)
```


## Recap

| Topic | API |
|---|---|
| Create queue | `get_or_create_annotation_queue(client, name=..., description=...)` |
| Route runs to queue | `create_run_rule(..., filter=..., add_to_annotation_queue_id=queue.id)` |
| Route failures only | filter includes `eq(feedback_key, "hallucination"), eq(feedback_score, 1)` |
| Manual push | `client.add_runs_to_annotation_queue(queue_id=..., run_ids=[...])` |

Labeled runs from this queue feed the datasets in Module 3 for the next iteration of experiments — closing
the build → trace → evaluate → review loop.
